# Automated Data Wrangling and Exploratory Data Analysis (EDA) with Python & Pandas

## This notebook serves as a high-level demonstration of AI-orchestrated data analytics. Every stage of this workflow—from initial script architecture to final visualization—is powered by Large Language Models (LLMs). By acting as the strategic lead and utilizing AI for code generation, we achieve a rapid, "zero-manual-coding" environment for complex data tasks.

Created for Prompt Engineering class.

### Step 1: Creating the tasks and lead the way

#### First prompt:

Output: (GPT-5.2-Codex)

#### Why This Prompt Works
 - Iterative Complexity: It forces the AI to think about the data lifecycle—from creation to cleaning to insight.
 - Agrotech Specificity: By mentioning "chemical dosage" and "wind speed," the code generated will include domain-specific logic (like drift loss) that a general prompt would miss.
 - Actionable Pandas Syntax: It requests specific library functions, ensuring the output isn't just theory, but executable code.

#### Review the output with another LLM this time with Gemini 3.1

#### With the rewrited code finnaly I can start coding

## Step 2: The agent sessions

After feeding this prompt to the agent, it generated a 450-line script that looked solid on the surface but contained some very basic errors.

After this simple path error the code looks great and usefull.

### Implementation: Farm-Specific Profiles in synthesize_data()

**Three Distinct Farm Profiles:**
- **F-100 (Struggling):** High dosage overcompensation (2.0 avg), expensive spray costs ($180-220), low yields (1500-2800 kg), high data quality issues
- **F-200 (Average):** Balanced dosage (1.2 avg), moderate costs ($140-180), good yields (3000-4500 kg)
- **F-300 (High Performer):** Efficient dosage (0.9 avg), low costs ($100-140), excellent yields (4000-5500 kg)

**Key Changes:**
- Per-farm distributions for dosage, cost, yield
- Farm-specific equipment reliability (F-100 uses older sprayer-x mostly)
- Variable data quality issues per farm (duplicates, missing values higher in F-100)
- Higher wind speed correlation with farm risk profiles

### Results & Outputs:

**Farm Risk Assessment (farm_risk.csv):**
- F-100: risk_score 5.58-6.36 (highest)
- F-200: risk_score 3.22-4.14 (moderate)
- F-300: risk_score 2.45-3.07 (lowest)

**Farm Profit Analysis (farm_profit.csv):**
- F-100: avg profit margin -0.06 to +0.44 (inconsistent, often unprofitable)
- F-200: avg profit margin 0.37-0.68 (solid, reliable)
- F-300: avg profit margin 0.50-0.73 (excellent, consistent)

**Cost Efficiency:**
- F-100: $90-184 per spray (expensive)
- F-200: $47-98 per spray (medium)
- F-300: $22-44 per spray (efficient)

**Generated Outputs:**
- 500-row realistic spray_logs_500.csv (3:3:1 farm distribution)
- 4 cleaned CSVs (spray_logs, weather, financials, metrics)
- 6 prediction CSVs (yield, profit, risk, dosage, equipment, crops)
- 10 visualization PNGs (farm comparisons, risk heatmap, equipment performance, etc.)

**Impact:** Clear differentiation enables stakeholder engagement and highlights intervention opportunities

## Agent Session Notes

I used the agent to read the pipeline, change the synthetic data generator, rerun the outputs, and verify the results. The main goal was to make the farm data feel less random and more like a real mix of bad, average, and strong farms.

### What the agent did
- Read `main.py` to find the data generation flow
- Rewrote `synthesize_data()` so each farm behaved differently
- Regenerated the raw spray log file
- Ran the pipeline, predictions, and visualizations
- Checked the outputs for risk and profit differences

## Bugs I Hit

### Bug #1: `pytest` could not find `main`
I hit a `ModuleNotFoundError` during test collection. The code was fine, but the test path was wrong. I fixed it by adding `tests/conftest.py` and putting the project root on `sys.path`.

**Status:** Fixed

### Bug #2: `merge_asof` wanted sorted keys
I tried to merge everything in one grouped call, but `merge_asof` was not happy with the sort order. I changed it to merge each field separately and then combine the results.

**Status:** Fixed

### Bug #3: Pandera warning about the old import
This was only a warning, not a blocker. Pandera was pointing at the old import style, so I left a note to switch to `import pandera.pandas as pa` later.

**Status:** Not urgent

### Bug #4: `polyfit` got mismatched arrays
The trend line code dropped NaNs separately for x and y, so the lengths no longer matched. I fixed it by building one shared mask before calling `np.polyfit`.

**Status:** Fixed

### Bug #5: Seaborn complained about `palette`
Seaborn changed the `boxplot` API, so the `palette` argument was the problem. I removed it and kept the default styling.

### Bug #6: The farms all looked the same
At first the synthetic generator was too flat. Every farm had the same kind of data, so the story was boring. I rewrote it with a bad farm, an average farm, and a strong farm so the outputs actually show a difference.

**Status:** Fixed

## Quick Wrap-Up

- I fixed 5 real bugs and left 1 warning as a follow-up.
- The agent work was mostly: read the code, change the generator, run the pipeline, and verify the outputs.
- The biggest lesson was that random data is not the same as realistic data.

### Bugs & Errors Encountered:

1. **ModuleNotFoundError in pytest** - `from main import ...` failed
   - **Fix:** Added conftest.py with sys.path management

2. **ValueError: left keys must be sorted in merge_asof**
   - **Cause:** Grouped merge_asof requires sorted keys, failed with multi-field groups
   - **Fix:** Switched to per-field merge strategy (iterate over field_ids)

3. **FutureWarning from Pandera deprecated import**
   - **Cause:** `import pandera as pa` (old style)
   - **Fix:** Non-blocking; can upgrade to `import pandera.pandas as pa`

4. **TypeError in trend line polyfit: mismatched array lengths**
   - **Cause:** Separate dropna on x and y arrays created length mismatch
   - **Fix:** Applied consistent NaN mask to both arrays before polyfit

5. **Seaborn palette parameter deprecated**
   - **Fix:** Removed palette argument from boxplot

## Phase 5: Farm Risk Profiling & Data Regeneration

### Prompt:
"Regenerate raw data and make more realistic, don't focus on every data type but create highly risky and bad farms"